# 01 – Chuẩn bị dữ liệu CREMA-D cho pilot (chạy một lần)
**Cài đặt notebook:** Accelerator = *None* (CPU là đủ); **Internet = On**.
**Cách chạy:** *Save Version → Save & Run All (Commit)*. Output `cremad_processed.tar` (khoảng 1,6 GB) được lưu cùng version; notebook `02_train_pilot` dùng nó làm input.

In [ ]:
# Đường dẫn (giữ nguyên trên Kaggle)
WORK = '/tmp'               # đĩa tạm của session: code, data giải nén, run
KIN  = '/kaggle/input'      # input gắn vào notebook (chỉ đọc)
KOUT = '/kaggle/working'    # output được lưu cùng version

In [ ]:
import os, glob, subprocess, sys, json, math, importlib
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
os.chdir(f'{WORK}/icme/pilot')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
print('CPUs:', os.cpu_count())
sh(f'df -h {WORK} {KOUT}', check=False)
for mod, pkg in [('librosa', 'librosa'), ('cv2', 'opencv-python-headless'), ('soundfile', 'soundfile')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [ ]:
# Tải khoảng 3 GB (WAV + FLV) qua media.githubusercontent.com và tiền xử lý giống OGM-GE.
# Raw đặt ở WORK và bị xóa sau khi xử lý.
sh(f'python prepare_cremad.py --root {WORK}/cremad --raw_dir {WORK}/cremad_raw --workers 16 --delete_raw')

In [ ]:
sh(f'python human_reliability.py --root {WORK}/cremad')
sh(f'python make_splits.py --root {WORK}/cremad')

In [ ]:
import pandas as pd
m = pd.read_csv(f'{WORK}/cremad/manifest.csv')
ok = (m.spec_ok == 1) & (m.n_frames > 0)
print('usable clips:', ok.sum(), '/', len(m), '(expected 7442 / 7442)')
print(m.loc[~ok].head(20))

In [ ]:
# Đóng gói thành 1 file để output gọn, tránh giới hạn số lượng file.
sh(f'tar -cf {KOUT}/cremad_processed.tar -C {WORK}/cremad spec frames_fps1 splits manifest.csv human_reliability.csv finishedResponses.csv')
sh(f'ls -la {KOUT}')